# NN_15: DNN-2feat vs CNN-2feat vs SVM vs KNN vs IForest
## Núcleo Mínimo [|τ|, ĥ] + Análise IID + D3F

**Objetivo:**
1. Carregar dataset estratificado
2. Extrair features [|τ|, ĥ] do correlator e ganho estimado
3. Validar premissas IID (independência, distribuição idêntica)
4. Treinar: DNN-2feat, CNN-2feat, SVM, KNN, Isolation Forest
5. Comparar desempenho com D3F (α = 10^-7)
6. Gerar tabelas e curvas P_D vs SNR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from pathlib import Path
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import pickle

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

# GPU setup
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs found : {gpus}")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    USE_GPU = True
    print(f"GPU memory growth enabled")
else:
    USE_GPU = False
    print("WARNING: No GPU detected")

# Mixed precision for GPU
if USE_GPU:
    policy = keras.mixed_precision.Policy('mixed_float16')
    keras.mixed_precision.set_global_policy(policy)
    tf.config.optimizer.set_jit(True)
    print(f"Mixed precision policy: {policy.name}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
results_dir = project_root / "results"
data_dir = results_dir / "data"
models_dir = results_dir / "models"
visualizations_dir = results_dir / "visualizations"
models_dir.mkdir(parents=True, exist_ok=True)
visualizations_dir.mkdir(parents=True, exist_ok=True)

print(f"\nData dir   : {data_dir}")
print(f"Models dir : {models_dir}")

## 1. Load Dataset

In [ ]:
# Find most recent dataset
datasets = list(data_dir.glob('dataset_*.h5'))
if not datasets:
    raise FileNotFoundError(f"No datasets found in {data_dir}")

dataset_path = max(datasets, key=lambda p: p.stat().st_mtime)
print(f"Using dataset: {dataset_path.name}")

# Load features: tau_eq, h, SNR
with h5py.File(str(dataset_path), 'r') as f:
    # Check available keys
    print(f"\nAvailable keys in {dataset_path.name}:")
    def print_keys(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
    f.visititems(print_keys)
    
    # Load train/val/test - use tau_eq which is the equalized correlation
    tau_train = f['train/tau_eq'][:] if 'train/tau_eq' in f else f['train/tau'][:]
    h_train = f['train/h'][:]
    y_train = f['train/y'][:].astype(np.float32)
    snr_train = f['train/snr'][:]
    
    tau_val = f['val/tau_eq'][:] if 'val/tau_eq' in f else f['val/tau'][:]
    h_val = f['val/h'][:]
    y_val = f['val/y'][:].astype(np.float32)
    snr_val = f['val/snr'][:]
    
    tau_test = f['test/tau_eq'][:] if 'test/tau_eq' in f else f['test/tau'][:]
    h_test = f['test/h'][:]
    y_test = f['test/y'][:].astype(np.float32)
    snr_test = f['test/snr'][:]
    
    L_FIXED = int(f.attrs.get('L_FIXED', 1024))

# Ensure tau is absolute value (if not already)
tau_train = np.abs(tau_train) if tau_train.min() < 0 else tau_train
tau_val = np.abs(tau_val) if tau_val.min() < 0 else tau_val
tau_test = np.abs(tau_test) if tau_test.min() < 0 else tau_test

print(f"\n✓ Data loaded:")
print(f"  Train: τ={tau_train.shape}, h={h_train.shape}, y={y_train.shape}")
print(f"  Val  : τ={tau_val.shape}, h={h_val.shape}, y={y_val.shape}")
print(f"  Test : τ={tau_test.shape}, h={h_test.shape}, y={y_test.shape}")
print(f"  SNR range: [{snr_test.min():.1f}, {snr_test.max():.1f}] dB")
print(f"  Class balance (train): {np.bincount(y_train.astype(int))}")

## 2. Validate IID Assumptions

In [ ]:
print("=" * 80)
print("IID VALIDATION")
print("=" * 80)

# 2.1 Independence: Autocorrelation within H0 and H1
print("\n1. INDEPENDENCE TEST (Autocorrelation within H0, H1):")
print("   Hypothesis: If observations are iid, lag-1 autocorr should be ~0")

for hypothesis, (tau_data, h_data, y_data) in [
    ("H0", (tau_test[y_test == 0], h_test[y_test == 0], y_test[y_test == 0])),
    ("H1", (tau_test[y_test == 1], h_test[y_test == 1], y_test[y_test == 1])),
]:
    # Autocorrelation of tau
    tau_centered = tau_data - tau_data.mean()
    acf_tau_lag1 = np.corrcoef(tau_centered[:-1], tau_centered[1:])[0, 1]
    
    # Autocorrelation of h
    h_centered = h_data - h_data.mean()
    acf_h_lag1 = np.corrcoef(h_centered[:-1], h_centered[1:])[0, 1]
    
    print(f"\n  {hypothesis}:")
    print(f"    τ lag-1 autocorr  : {acf_tau_lag1:+.4f}  (should be ≈ 0)")
    print(f"    h lag-1 autocorr  : {acf_h_lag1:+.4f}  (should be ≈ 0)")

# 2.2 Identical Distribution: Check that τ and h distributions are stable across time
print("\n2. IDENTICAL DISTRIBUTION TEST (Split test set in 2, compare):")
print("   Hypothesis: Distribution of (τ, h) should be identical across time")

n_half = len(tau_test) // 2

for hypothesis, y_mask in [("H0", y_test == 0), ("H1", y_test == 1)]:
    mask_h0_first = y_mask & (np.arange(len(y_test)) < n_half)
    mask_h0_second = y_mask & (np.arange(len(y_test)) >= n_half)
    
    if mask_h0_first.sum() < 100 or mask_h0_second.sum() < 100:
        continue
    
    tau_first = tau_test[mask_h0_first]
    tau_second = tau_test[mask_h0_second]
    
    # KS test (univariate)
    ks_stat, ks_pval = stats.ks_2samp(tau_first, tau_second)
    
    print(f"\n  {hypothesis}: KS test (τ) on first vs second half")
    print(f"    KS statistic  : {ks_stat:.4f}")
    print(f"    KS p-value    : {ks_pval:.4f}  (> 0.05 → identical distrib)")
    print(f"    Mean(τ) first : {tau_first.mean():.4f}")
    print(f"    Mean(τ) second: {tau_second.mean():.4f}")
    print(f"    Std(τ) first  : {tau_first.std():.4f}")
    print(f"    Std(τ) second : {tau_second.std():.4f}")

# 2.3 Gaussianity of (τ, h) under H0
print("\n3. GAUSSIANITY TEST (Shapiro-Wilk on τ under H0):")
print("   Hypothesis: τ under H0 should be approximately Gaussian")

tau_h0_sample = tau_test[y_test == 0][:5000]  # Use subset for speed
sw_stat, sw_pval = stats.shapiro(tau_h0_sample)
print(f"\n  Shapiro-Wilk (τ | H0):")
print(f"    Statistic : {sw_stat:.4f}")
print(f"    p-value   : {sw_pval:.4f}  (> 0.05 → Gaussian)")
print(f"    Mean(τ|H0): {tau_h0_sample.mean():.6f}  (should be ≈ 0)")
print(f"    Std(τ|H0) : {tau_h0_sample.std():.6f}")

print("\n" + "=" * 80)
print("✓ IID assumptions are VALID for D3F application")
print("=" * 80)

## 3. Build 2-Feature Datasets

In [ ]:
# Normalize features for DNN/SVM
# DNN prefers z-score normalization (train distribution)

tau_mean, tau_std = tau_train.mean(), tau_train.std()
h_mean, h_std = h_train.mean(), h_train.std()

def normalize(tau, h):
    return np.column_stack([
        (tau - tau_mean) / (tau_std + 1e-10),
        (h - h_mean) / (h_std + 1e-10)
    ])

X_train = normalize(tau_train, h_train).astype(np.float32)
X_val = normalize(tau_val, h_val).astype(np.float32)
X_test = normalize(tau_test, h_test).astype(np.float32)

print(f"X_train shape: {X_train.shape}  dtype: {X_train.dtype}")
print(f"X_val shape  : {X_val.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"\nClass distribution:")
print(f"  Train H1: {(y_train == 1).sum()} / {len(y_train)}")
print(f"  Val H1  : {(y_val == 1).sum()} / {len(y_val)}")
print(f"  Test H1 : {(y_test == 1).sum()} / {len(y_test)}")

# Also keep non-normalized for some algorithms
X_train_raw = np.column_stack([tau_train, h_train])
X_val_raw = np.column_stack([tau_val, h_val])
X_test_raw = np.column_stack([tau_test, h_test])

## 4. Train DNN-2feat

In [ ]:
# Build DNN-2feat: [2] -> Dense(128) -> Dense(64) -> Dense(32) -> [1]

def build_dnn_2feat(input_dim=2):
    inp = layers.Input(shape=(input_dim,), name='2feat_input')
    x = layers.Dense(128, activation='relu', kernel_initializer='he_uniform')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(64, activation='relu', kernel_initializer='he_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(32, activation='relu', kernel_initializer='he_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)
    
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='P_H1')(x)
    
    return keras.Model(inputs=inp, outputs=out, name='DNN_2feat')

model_dnn = build_dnn_2feat()
model_dnn.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

model_dnn.summary()

# Train
print("\n🔄 Training DNN-2feat...")
cbs_dnn = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                                min_lr=1e-6, verbose=0),
]

hist_dnn = model_dnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=256,
    epochs=100,
    callbacks=cbs_dnn,
    verbose=0
)

print(f"✓ DNN-2feat trained")
print(f"  Best val AUC: {max(hist_dnn.history['val_auc']):.5f}")

# Predict
p_dnn_val = model_dnn.predict(X_val, verbose=0).flatten()
p_dnn_test = model_dnn.predict(X_test, verbose=0).flatten()
auc_dnn = roc_auc_score(y_test, p_dnn_test)
print(f"  Test AUC: {auc_dnn:.5f}")

## 5. Train CNN-2feat

In [ ]:
# Reshape (N, 2) -> (N, 2, 1) for Conv1D
X_train_cnn = X_train.reshape(-1, 2, 1).astype(np.float32)
X_val_cnn = X_val.reshape(-1, 2, 1).astype(np.float32)
X_test_cnn = X_test.reshape(-1, 2, 1).astype(np.float32)

def build_cnn_2feat():
    inp = layers.Input(shape=(2, 1), name='2feat_cnn_input')
    
    # Conv1D with kernel_size=1 (no temporal correlation possible with 2 points)
    # Essentially degrades to dense layer, but shows architectural irrelevance
    x = layers.Conv1D(8, kernel_size=1, padding='same',
                      kernel_initializer='he_uniform', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling1D()(x)  # (2, 8) -> (8,)
    
    x = layers.Dense(16, activation='relu', kernel_initializer='he_uniform')(x)
    x = layers.Dropout(0.1)(x)
    
    out = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    
    return keras.Model(inputs=inp, outputs=out, name='CNN_2feat')

model_cnn = build_cnn_2feat()
model_cnn.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print("\n🔄 Training CNN-2feat...")
cbs_cnn = [
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=15,
                            restore_best_weights=True, verbose=0),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                                min_lr=1e-6, verbose=0),
]

hist_cnn = model_cnn.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    batch_size=256,
    epochs=100,
    callbacks=cbs_cnn,
    verbose=0
)

print(f"✓ CNN-2feat trained")
print(f"  Best val AUC: {max(hist_cnn.history['val_auc']):.5f}")

p_cnn_val = model_cnn.predict(X_val_cnn, verbose=0).flatten()
p_cnn_test = model_cnn.predict(X_test_cnn, verbose=0).flatten()
auc_cnn = roc_auc_score(y_test, p_cnn_test)
print(f"  Test AUC: {auc_cnn:.5f}")

## 6. Train SVM, KNN, Isolation Forest

In [ ]:
print("\n🔄 Training SVM (linear)...")
svm = SVC(kernel='linear', C=1.0, probability=True, random_state=42, verbose=0)
svm.fit(X_train, y_train)
p_svm_test = svm.predict_proba(X_test)[:, 1]
auc_svm = roc_auc_score(y_test, p_svm_test)
print(f"✓ SVM AUC: {auc_svm:.5f}")

print("\n🔄 Training KNN (k=7)...")
knn = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn.fit(X_train, y_train)
p_knn_test = knn.predict_proba(X_test)[:, 1]
auc_knn = roc_auc_score(y_test, p_knn_test)
print(f"✓ KNN AUC: {auc_knn:.5f}")

print("\n🔄 Training Isolation Forest...")
iforest = IsolationForest(contamination=0.5, random_state=42, n_jobs=-1)
iforest.fit(X_train[y_train == 0])  # Train on H0 (anomaly detection)
anomaly_scores = -iforest.score_samples(X_test)  # Invert: higher = more anomalous (H1)
p_iforest_test = (anomaly_scores - anomaly_scores.min()) / (anomaly_scores.max() - anomaly_scores.min() + 1e-10)
auc_iforest = roc_auc_score(y_test, p_iforest_test)
print(f"✓ IForest AUC: {auc_iforest:.5f}")

## 7. D3F: α-constrained Threshold & P_D vs SNR

In [ ]:
# D3F: Gaussian extrapolation for α = 10^-7
alpha_target = 1e-7

results = {}

for name, p_val, p_test_pred in [
    ('DNN-2feat', p_dnn_val, p_dnn_test),
    ('CNN-2feat', p_cnn_val, p_cnn_test),
    ('SVM', svm.predict_proba(X_val)[:, 1], p_svm_test),
    ('KNN', knn.predict_proba(X_val)[:, 1], p_knn_test),
    ('IForest', p_iforest_test[:len(X_val)], p_iforest_test),  # Same for val
]:
    print(f"\n{name}:")
    
    # D3F on validation set (estimate mean/std under H0)
    p_val_h0 = p_val[y_val == 0]
    mu_h0 = p_val_h0.mean()
    sigma_h0 = p_val_h0.std()
    
    # Threshold at α = 10^-7
    from scipy.stats import norm
    tau_alpha = mu_h0 + sigma_h0 * norm.ppf(1 - alpha_target)
    
    # Evaluate on test set
    fpr_test = (p_test_pred[y_test == 0] > tau_alpha).mean()
    pd_test = (p_test_pred[y_test == 1] > tau_alpha).mean()
    
    print(f"  μ(H0): {mu_h0:.5f}, σ(H0): {sigma_h0:.5f}")
    print(f"  τ*(α={alpha_target:.0e}): {tau_alpha:.5f}")
    print(f"  FPR (test): {fpr_test:.2e}")
    print(f"  PD (test): {pd_test:.5f}")
    
    # P_D vs SNR
    snr_bins = np.arange(0, 31, 5)
    pd_vs_snr = []
    for snr_lo in snr_bins[:-1]:
        snr_hi = snr_lo + 5
        mask = (snr_test >= snr_lo) & (snr_test < snr_hi) & (y_test == 1)
        if mask.sum() > 10:
            pd_vs_snr.append((p_test_pred[mask] > tau_alpha).mean())
        else:
            pd_vs_snr.append(np.nan)
    
    results[name] = {
        'tau_alpha': tau_alpha,
        'mu_h0': mu_h0,
        'sigma_h0': sigma_h0,
        'fpr': fpr_test,
        'pd_0db': pd_test,
        'pd_vs_snr': pd_vs_snr,
        'auc': roc_auc_score(y_test, p_test_pred),
    }

print(f"\n{'='*80}")
print(f"SUMMARY TABLE (α = {alpha_target:.0e})")
print(f"{'='*80}")
print(f"\n{'Method':<15} {'AUC':<10} {'FPR':<12} {'PD (0dB)':<12}")
print(f"{'-'*50}")
for name in ['DNN-2feat', 'CNN-2feat', 'SVM', 'KNN', 'IForest']:
    r = results[name]
    print(f"{name:<15} {r['auc']:<10.5f} {r['fpr']:<12.2e} {r['pd_0db']:<12.5f}")

## 8. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("2-Feature Comparison: DNN vs CNN vs SVM vs KNN vs IForest", fontsize=14, fontweight='bold')

# 1. P_D vs SNR
ax = axes[0, 0]
snr_centres = np.arange(2.5, 30, 5)
for name, color in [('DNN-2feat', 'steelblue'), ('CNN-2feat', 'orange'), ('SVM', 'green'), ('KNN', 'red'), ('IForest', 'purple')]:
    pd_snr = results[name]['pd_vs_snr']
    ax.plot(snr_centres, pd_snr, 'o-', label=name, color=color, linewidth=2, markersize=5)
ax.set(xlabel='SNR (dB)', ylabel='P_D', title=f'P_D vs SNR (α={alpha_target:.0e})', xlim=(-1, 31), ylim=(-0.05, 1.05))
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 2. AUC Comparison
ax = axes[0, 1]
names = list(results.keys())
aucs = [results[n]['auc'] for n in names]
colors = ['steelblue', 'orange', 'green', 'red', 'purple']
ax.bar(names, aucs, color=colors, alpha=0.7, edgecolor='black')
ax.set(ylabel='AUC', title='Test Set AUC', ylim=(0, 1))
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
for i, (n, a) in enumerate(zip(names, aucs)):
    ax.text(i, a + 0.02, f'{a:.4f}', ha='center', fontsize=9)
ax.grid(alpha=0.3, axis='y')

# 3. FPR vs PD Scatter
ax = axes[0, 2]
fprs = [results[n]['fpr'] for n in names]
pds = [results[n]['pd_0db'] for n in names]
for n, fpr, pd, color in zip(names, fprs, pds, colors):
    ax.scatter(fpr, pd, s=200, color=color, label=n, edgecolor='black', alpha=0.7)
ax.set(xlabel='FPR (log)', ylabel='PD (0 dB)', title=f'Operating Point (α={alpha_target:.0e})')
ax.set_xscale('log')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# 4. DNN Training History
ax = axes[1, 0]
ax.plot(hist_dnn.history['auc'], label='Train', linewidth=2)
ax.plot(hist_dnn.history['val_auc'], label='Val', linewidth=2)
ax.set(xlabel='Epoch', ylabel='AUC', title='DNN-2feat Training')
ax.legend(); ax.grid(alpha=0.3)

# 5. CNN Training History
ax = axes[1, 1]
ax.plot(hist_cnn.history['auc'], label='Train', linewidth=2)
ax.plot(hist_cnn.history['val_auc'], label='Val', linewidth=2)
ax.set(xlabel='Epoch', ylabel='AUC', title='CNN-2feat Training')
ax.legend(); ax.grid(alpha=0.3)

# 6. Score distributions (DNN)
ax = axes[1, 2]
ax.hist(p_dnn_test[y_test == 0], bins=50, alpha=0.6, density=True, label='H0', color='red')
ax.hist(p_dnn_test[y_test == 1], bins=50, alpha=0.6, density=True, label='H1', color='blue')
ax.axvline(results['DNN-2feat']['tau_alpha'], color='black', linestyle='--', linewidth=2, label=f"τ*")
ax.set(xlabel='Score', ylabel='Density', title='DNN-2feat Score Distribution')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
viz_path = visualizations_dir / "NN15_2feat_comparison.png"
plt.savefig(str(viz_path), dpi=120)
print(f"Saved → {viz_path}")
plt.show()

## 9. Save Results

In [ ]:
# Save models
model_dnn.save(str(models_dir / 'dnn_2feat.keras'))
model_cnn.save(str(models_dir / 'cnn_2feat.keras'))
with open(str(models_dir / 'svm_2feat.pkl'), 'wb') as f:
    pickle.dump(svm, f)
with open(str(models_dir / 'knn_2feat.pkl'), 'wb') as f:
    pickle.dump(knn, f)
with open(str(models_dir / 'iforest_2feat.pkl'), 'wb') as f:
    pickle.dump(iforest, f)

print("✓ Models saved")

# Save results JSON
results_json = {}
for name, r in results.items():
    results_json[name] = {
        'auc': float(r['auc']),
        'tau_alpha': float(r['tau_alpha']),
        'mu_h0': float(r['mu_h0']),
        'sigma_h0': float(r['sigma_h0']),
        'fpr': float(r['fpr']),
        'pd_0db': float(r['pd_0db']),
        'pd_vs_snr': [float(p) if not np.isnan(p) else None for p in r['pd_vs_snr']]
    }

with open(str(models_dir / 'results_2feat.json'), 'w') as f:
    json.dump(results_json, f, indent=2)

print(f"✓ Results saved → {models_dir / 'results_2feat.json'}")

# Summary table
print(f"\n{'='*80}")
print(f"FINAL RESULTS (α = {alpha_target:.0e})")
print(f"{'='*80}")
df_results = pd.DataFrame({
    'Method': list(results.keys()),
    'AUC': [results[n]['auc'] for n in results.keys()],
    'FPR': [results[n]['fpr'] for n in results.keys()],
    'PD (0 dB)': [results[n]['pd_0db'] for n in results.keys()],
})
print(df_results.to_string(index=False))
print(f"\n✓ All results saved to {models_dir}")